# Job Salary ML Model

The purpose of this program is to create a model that can predict the salary for job postings in the United States.

# Data Loading

### Data Paths

In [3]:
%spark
import org.apache.spark.sql.types._
import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.functions._

val ILOPath = "ILOData.parquet"
val OEPath = "cleanedOEData.parquet"

In [4]:
%spark
val ILODF = spark.read
    .load(ILOPath)
    .withColumnRenamed("sex.label", "sex")
    .withColumnRenamed("classif1.label", "class")

z.show(ILODF)

In [5]:
%spark
ILODF.printSchema()

z.show(ILODF.summary())

In [6]:
%spark
z.show(ILODF.describe())

In [7]:
%spark
val ILODF2 = ILODF.select("class").distinct.orderBy("class")

z.show(ILODF2)

In [8]:
%spark
val OEDF = spark.read.load(OEPath)

z.show(OEDF)

In [9]:
%spark
z.show(OEDF.select("occupation_category_name").distinct)

### Employment Model

In [11]:
%spark
val ILODF

### Wage Model

In [ ]:
%spark
val wageOEDF = OEDF.filter($"areatype_name" === "Metropolitan or nonmetropolitan area")
    .drop("areatype_name")
    .drop("industry_name")
    .filter($"occupation_description".isNotNull)
    .drop("occupation_description")
    .filter($"datatype_name" === "Annual mean wage")
    .drop("datatype_name")
    .drop("sector_name")
    .withColumn("log_salary", log($"value"))
    .withColumnRenamed("value", "Annual Mean Wage")

z.show(wageOEDF)

In [14]:
%spark
val Array(trainWageOEDF, testWageOEDF) = wageOEDF.randomSplit(Array(.8, .2), seed=42)
println(f"There are ${trainWageOEDF.cache().count()} rows in the training set, and ${testWageOEDF.cache().count()} in the test set")

In [15]:
%spark
import org.apache.spark.ml.feature.{OneHotEncoder, StringIndexer}
val categoricalOECols = trainWageOEDF.dtypes.filter(_._2 == "StringType").map(_._1)
val indexOutputOECols = categoricalOECols.map(_ + "_index")
val oheOutputOECols = categoricalOECols.map(_ + "_OHE")

val stringIndexerOE = new StringIndexer()
  .setInputCols(categoricalOECols)
  .setOutputCols(indexOutputOECols)
  .setHandleInvalid("skip")

val oheEncoderOE = new OneHotEncoder()
  .setInputCols(indexOutputOECols)
  .setOutputCols(oheOutputOECols)

In [16]:
%spark
import org.apache.spark.ml.feature.VectorAssembler

val vecAssemblerOE = new VectorAssembler()
  .setInputCols(oheOutputOECols)
  .setOutputCol("features")

In [17]:
%spark
import org.apache.spark.ml.regression.LinearRegression

val lrOE = new LinearRegression()
  .setLabelCol("log_salary")
  .setFeaturesCol("features")

In [18]:
%spark
import org.apache.spark.ml.Pipeline

val pipelineOE = new Pipeline()
  .setStages(Array(stringIndexerOE, oheEncoderOE, vecAssemblerOE, lrOE))

val pipelineModelOE = pipelineOE.fit(trainWageOEDF)
val predOEDF = pipelineModel.transform(testWageOEDF)
z.show(predOEDF.select("features", "log_salary", "prediction"))

In [19]:
%spark
val predFinalOEDF = predOEDF.withColumn("pred_salary", exp($"prediction"))

z.show(predFinalOEDF.select("features","Annual Mean Wage","pred_salary"))

In [20]:
%spark
import org.apache.spark.ml.evaluation.RegressionEvaluator
val regressionEvaluatorOE = new RegressionEvaluator()
.setPredictionCol("pred_salary")
.setLabelCol("Annual Mean Wage")
.setMetricName("rmse")
val rmseOE = regressionEvaluatorOE.evaluate(predFinalOEDF)

println(f"RMSE is $rmseOE%.1f")

In [21]:
%spark
val r2OE = regressionEvaluatorOE.setMetricName("r2").evaluate(predFinalOEDF)
println(s"R2 is $r2OE")